# Setup

In [ ]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = "retina"

In [ ]:
import os
import shutil
from functools import partial
from pprint import pprint
from typing import Callable, cast

from tqdm.auto import tqdm

tqdm.pandas()
os.chdir(os.path.abspath(os.path.join(os.getcwd(), "..", "..")))
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from experiments.data_preparation.src import (  # noqa: E402
    add_token_counts_and_filter,
    compute_budgets,
    compute_multi_turn_token_counts,
    get_df_stats,
    get_df_stats__by_source,
    get_device,
    get_token_model,
    nlp_quality_filter,
    normalize_original_audio_entry,
    plot_token_count_comparison,
    prepare_for_save,
    save_dataset_and_sample,
    stratified_sample,
    synthesize_voices,
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from datasets import DatasetDict

np.random.seed(0)

In [ ]:
device = get_device()
print(f"Using device: {device}")

In [ ]:
from experiments.data_preparation.constants import (
    TTS_CONFIGS,
    VOICE_BENCH__CONFIG,
    TTSConfig,
)
from experiments.data_preparation.io import load_dataset as src_load_dataset
from experiments.data_preparation.languages import LanguageClassifier
from experiments.data_preparation.nlp import TTS, split_into_sentences

In [ ]:
DATASET_NAMES_TO_SKIP: list[str] = ["MT-Bench"]  # multi turn
MAX_SENTENCES: int = 8
NUM_SAMPLES_1K: int = 10000
NUM_SAMPLES_100: int = 100

tts: TTS = TTS()
language_classifier = LanguageClassifier()
load_dataset: Callable[[str], DatasetDict] = partial(
    src_load_dataset, config=VOICE_BENCH__CONFIG
)

tts_configs: dict[str, TTSConfig] = TTS_CONFIGS["en"]
male_config: TTSConfig = TTS_CONFIGS["en"]["male"]
female_config: TTSConfig = TTS_CONFIGS["en"]["female"]

In [ ]:
model_for_tokens = get_token_model(device)

# Text dataset preparation and EDA

## Preprocessing

### Dataset preparation

Load all relevant data. We preserve `audio__original` from the source dataset alongside each entry and immediately normalize it to the same byte-container format as synthetic audio columns (`list[bytes]`) with a matching `audio__original__duration` (`list[float]`).

In [ ]:
dt: list[tuple[str, str, str, dict]] = []
for df_name, config_name in VOICE_BENCH__CONFIG.configs.items():
    if df_name in DATASET_NAMES_TO_SKIP:
        continue

    print(f"Processing {df_name}...")

    splits = load_dataset(config_name)
    for split_name, split_ds in splits.items():
        print(f"\tSplit: {split_name} ({split_ds.num_rows} rows)")
        dt.extend(
            [
                cast(
                    tuple[str, str, str, dict],
                    [entry["prompt"], df_name, split_name, entry["audio"]],
                )
                for entry in split_ds
            ]
        )

df = pd.DataFrame(dt, columns=["prompt", "dataset", "split", "audio__original"])
del dt

In [ ]:
normalized_original = df["audio__original"].progress_apply(
    normalize_original_audio_entry
)
df["audio__original"] = normalized_original.apply(lambda x: x[0])
df["audio__original__duration"] = normalized_original.apply(lambda x: x[1])

del normalized_original

Split prompt into sentences and check if they are English.

In [ ]:
df["sentences"] = df["prompt"].progress_apply(split_into_sentences)
df["sentences__num"] = df["sentences"].apply(len)
df["is_english"] = df["prompt"].progress_apply(language_classifier.is_english)

In [ ]:
pprint(get_df_stats(df, include_audio=False))

In [ ]:
non_english_entries: pd.Series = df[~df["is_english"]]["prompt"]
pprint(sorted(non_english_entries.unique()))

In [ ]:
print(pd.Series(non_english_entries).value_counts())
del non_english_entries

Deduplicate by prompt. Because many sub-datasets share identical prompts encoded with different TTS models, we collapse duplicates and keep the union of source dataset names. For `audio__original` and `audio__original__duration` we retain the first occurrence.

In [ ]:
df = (
    df.groupby(["prompt"])
    .agg(
        {
            "dataset": lambda x: sorted(set(x)),
            "sentences": "first",
            "sentences__num": "first",
            "audio__original": "first",
            "audio__original__duration": "first",
        }
    )
    .reset_index()
    .rename(columns={"dataset": "datasets"})
)

# Single Sentence Dataset (100 samples)

A dataset containing single-sentence scenarios with synthesized male/female audio.

## 1 · Filter to single-sentence entries

In [ ]:
single_sentence__df: pd.DataFrame = df[df["sentences__num"] == 1].copy()
print(f"Number of unique single-sentence entries: {len(single_sentence__df)}")

In [ ]:
single_sentence__df.head(3)

In [ ]:
pprint(single_sentence__df["datasets"].apply(len).value_counts().sort_index())

## 2 · Token-count filter

Count explainability tokens and keep entries with ≤ 10 tokens.

In [ ]:
single_sentence__df, candidates = add_token_counts_and_filter(
    single_sentence__df, model_for_tokens, max_token_count=10
)

## 3 · Stratified sampling

Stratify by dataset to reduce to 50%, then stratify by token_count to 100 samples.

In [ ]:
candidates["datasets__combined"] = candidates["datasets"].str.join(" ")

single_sentence__df = (
    candidates.groupby("datasets__combined", group_keys=False)
    .apply(lambda x: x.sample(frac=0.5, random_state=0), include_groups=False)
    .reset_index(drop=True)
)
print(f"Size after dataset stratification: {len(single_sentence__df)}")

In [ ]:
subset_low_tokens = stratified_sample(
    pool=single_sentence__df,
    n_target=NUM_SAMPLES_100,
    strat_col="token_count",
)
print("Selected subset size:", len(subset_low_tokens))
print("Token count summary (selected subset):")
print(subset_low_tokens["token_count"].describe())

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.histplot(single_sentence__df["token_count"], bins=30, kde=False)
plt.title("Token counts — after dataset stratification")
plt.xlabel("token_count")

plt.subplot(1, 2, 2)
sns.histplot(subset_low_tokens["token_count"], bins=30, color="orange", kde=False)
plt.title("Token counts — final 100-sample dataset")
plt.xlabel("token_count")
plt.tight_layout()
plt.show()

In [ ]:
single_sentence__df = subset_low_tokens

## 4 · TTS audio synthesis

In [ ]:
single_sentence__df = await synthesize_voices(
    single_sentence__df, tts=tts, male_config=male_config, female_config=female_config
)

## Stats

In [ ]:
get_df_stats__by_source(single_sentence__df)

In [ ]:
pprint(get_df_stats(single_sentence__df))

In [ ]:
pprint(compute_budgets(single_sentence__df["token_count"]))

## 5 · Save dataset

In [ ]:
single_sentence__df__to_save = single_sentence__df.drop(
    columns=["prompt", "sentences__num"]
)
save_dataset_and_sample(
    single_sentence__df__to_save,
    VOICE_BENCH__CONFIG.data_dir,
    "single_sentence",
)
single_sentence__df__to_save.head(3)

## Section cleanup

In [ ]:
del single_sentence__df, single_sentence__df__to_save, candidates, subset_low_tokens

# Single Sentence Dataset (1 000 samples with NLP filtering)

A higher-quality dataset built with embedding-based interestingness scoring and semantic deduplication.
Contains three audio columns: `audio__original`, `audio__male`, `audio__female`.

## 1 · Filter to single-sentence entries

In [ ]:
single_sentence__df: pd.DataFrame = df[df["sentences__num"] == 1].copy()
print(
    f"Unique single-sentence entries before quality filtering: {len(single_sentence__df)}"
)

In [ ]:
single_sentence__df.head(3)

## 2 · NLP quality & interestingness filter

We use `sentence-transformers` (`all-MiniLM-L6-v2`) to compute dense embeddings for every candidate sentence and then:

1. **Hard rule-based pre-filter** — discard trivially short / low-information entries.
2. **Embedding-based diversity / interestingness score** — project embeddings onto a "general-purpose query" anchor vector and keep the top-scoring fraction.
3. **Semantic deduplication** — greedy nearest-neighbour pass with a cosine-similarity threshold.

In [ ]:
single_sentence__df, embeddings = nlp_quality_filter(
    single_sentence__df, device=str(device)
)

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(single_sentence__df["interestingness_score"], bins=40, kde=True)
plt.title("Interestingness score distribution after NLP filtering")
plt.xlabel("cosine similarity to anchor centroid")
plt.tight_layout()
plt.show()

## 3 · Token-count filter

Count explainability tokens (mask sum) and keep only entries with ≤ 10 tokens.

In [ ]:
single_sentence__df, candidates = add_token_counts_and_filter(
    single_sentence__df, model_for_tokens, max_token_count=10
)

## 4 · Stratified sampling to 1 000 samples

In [ ]:
candidates["datasets__combined"] = candidates["datasets"].apply(lambda x: " ".join(x))

In [ ]:
single_sentence__df = stratified_sample(
    pool=candidates,
    n_target=NUM_SAMPLES_1K,
    strat_col="token_count",
)
print(f"Final dataset size: {len(single_sentence__df)}")

In [ ]:
plot_token_count_comparison(candidates, single_sentence__df, NUM_SAMPLES_1K)

## 5 · TTS audio synthesis

Generate `audio__male` and `audio__female` alongside the already-present `audio__original`.

In [ ]:
single_sentence__df = await synthesize_voices(
    single_sentence__df, tts=tts, male_config=male_config, female_config=female_config
)

## Stats

In [ ]:
get_df_stats__by_source(single_sentence__df)

In [ ]:
single_sentence__df = pd.read_parquet(
    VOICE_BENCH__CONFIG.data_dir / "single_sentence_1k.parquet"
)

In [ ]:
pprint(get_df_stats(single_sentence__df))

In [ ]:
pprint(compute_budgets(single_sentence__df["token_count"]))

## 6 · Save dataset

The saved parquet contains audio columns `audio__original`, `audio__male`, `audio__female` and `audio__original__duration`.

In [ ]:
single_sentence__df__to_save = prepare_for_save(single_sentence__df)

single_sentence__df__to_save["sentences"] = single_sentence__df__to_save[
    "prompt"
].progress_apply(lambda x: [x])
single_sentence__df__to_save.drop(columns=["prompt"], inplace=True)

save_dataset_and_sample(
    single_sentence__df__to_save,
    VOICE_BENCH__CONFIG.data_dir,
    "single_sentence_1k",
)
single_sentence__df__to_save.head(3)

In [ ]:
single_sentence__df__to_save.head(3)

## Section cleanup

In [ ]:
del single_sentence__df, single_sentence__df__to_save, candidates, embeddings

# Multi Sentence Dataset (100 samples)

Dataset containing samples with 2–8 sentences, synthesized as multi-turn conversations.

In [ ]:
multi_sentence__df: pd.DataFrame = df[df["sentences__num"] > 1].copy()
print(f"Number of unique multi-sentence entries: {len(multi_sentence__df)}")

In [ ]:
multi_sentence__df.head(3)

In [ ]:
pprint(multi_sentence__df["datasets"].apply(len).value_counts().sort_index())

In [ ]:
pprint(multi_sentence__df["sentences__num"].value_counts().sort_index())

Drop entries with more than 8 sentences.

In [ ]:
multi_sentence__df = multi_sentence__df[
    multi_sentence__df["sentences__num"] <= MAX_SENTENCES
].copy()

## Token counting

Count multi-turn tokens and keep entries with ≤ 30 tokens.

In [ ]:
multi_sentence__df["token_count"] = compute_multi_turn_token_counts(
    multi_sentence__df, model=model_for_tokens, sentences_column="sentences"
)
print(f"Computed token counts for {len(multi_sentence__df)} entries.")

In [ ]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.histplot(multi_sentence__df["token_count"], bins=30, kde=False)
plt.title("Token counts across multi_sentence__df")
plt.xlabel("token_count")

In [ ]:
MAX_TOKEN_COUNT_MS = 30
multi_sentence__df = multi_sentence__df[
    multi_sentence__df["token_count"] <= MAX_TOKEN_COUNT_MS
].copy()
print(f"Candidates with token_count <= {MAX_TOKEN_COUNT_MS}: {len(multi_sentence__df)}")

## Stratified sampling

Stratify by dataset+sentence_count to 35%, then by token_count to 100 samples.

In [ ]:
multi_sentence__df["datasets__combined"] = multi_sentence__df["datasets"].str.join(" ")
multi_sentence__df["sentences__num__grp"] = multi_sentence__df["sentences__num"]
multi_sentence__df = multi_sentence__df.groupby(
    ["datasets__combined", "sentences__num__grp"], group_keys=False
).apply(lambda x: x.sample(frac=0.35, random_state=0), include_groups=False)
print(f"Size after dataset×sentence stratification: {len(multi_sentence__df)}")

In [ ]:
multi_sentence__df = stratified_sample(
    pool=multi_sentence__df.reset_index(drop=True),
    n_target=NUM_SAMPLES_100,
    strat_col="token_count",
)
print("Selected subset size:", len(multi_sentence__df))
print("Token count summary:")
print(multi_sentence__df["token_count"].describe())

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.histplot(multi_sentence__df["token_count"], bins=30, kde=False)
plt.title("Token counts — multi-sentence final")
plt.xlabel("token_count")
plt.tight_layout()
plt.show()

In [ ]:
pprint(multi_sentence__df["sentences__num"].value_counts().sort_index())

## TTS audio synthesis

In [ ]:
multi_sentence__df = await synthesize_voices(
    multi_sentence__df, tts=tts, male_config=male_config, female_config=female_config
)

## Stats

In [ ]:
get_df_stats__by_source(multi_sentence__df)

In [ ]:
pprint(get_df_stats(multi_sentence__df))

In [ ]:
pprint(compute_budgets(multi_sentence__df["token_count"]))

## Save dataset

In [ ]:
multi_sentence__df__to_save = multi_sentence__df.drop(columns=["prompt"])
save_dataset_and_sample(
    multi_sentence__df__to_save,
    VOICE_BENCH__CONFIG.data_dir,
    "multi_sentence",
)

In [ ]:
multi_sentence__df__to_save.head(3)

## Section cleanup

In [ ]:
del multi_sentence__df, multi_sentence__df__to_save, df

# Final cleanup

In [ ]:
shutil.rmtree(VOICE_BENCH__CONFIG.cache_dir)